# 08 — Guided Mini-Project (Reference Solution)

**Environment:** Ubuntu 22.04 · Python 3.12.3 · OpenJDK 17 · Spark 4.1.x · local PySpark.

Run cells in order. Each notebook can start a Spark session independently.

**Use after the participant exercise.** Full end-to-end implementation including validations and Parquet export.

In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'data' / 'transactions.csv').is_file()), None)
assert ROOT is not None, 'Open the workshop folder in Jupyter before running this notebook.'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from training_lib import create_spark, read_raw, typed_transactions, quality_condition, clean_transactions, completed_enriched, revenue_by_category
from pyspark.sql import functions as F
spark = create_spark('TG117-Workshop')
print('Spark:', spark.version, '| Python:', sys.version.split()[0], '| master:', spark.sparkContext.master)

In [ ]:
raw, customers, products = read_raw(spark, ROOT)
assert raw.count() == 58
raw.groupBy('transaction_id').count().filter(F.col('count') > 1).show()

In [ ]:
typed = typed_transactions(raw)
rejected = typed.filter(~quality_condition())
clean = clean_transactions(raw)
assert typed.count()==57
assert rejected.count()==7
assert clean.count()==50
print('Unique rows:',typed.count(),'rejected:',rejected.count(),'clean:',clean.count())

In [ ]:
completed = clean.filter(F.col('status') == 'COMPLETED')
product_rejects = completed.join(products.select('product_id'),'product_id','left_anti')
customer_rejects = completed.join(customers.select('customer_id'),'customer_id','left_anti')
assert product_rejects.count()==1 and customer_rejects.count()==1
enriched = completed_enriched(clean,customers,products)
assert enriched.count()==36
print('Completed, reference-valid:',enriched.count())

In [ ]:
report = revenue_by_category(enriched)
report.show(truncate=False)
assert report.count()==4
assert str(report.agg(F.sum('revenue')).first()[0])=='342228.00'
expected={'Accessories':('16','52574.00'), 'Audio':('4','20992.00'), 'Electronics':('12','265968.00'), 'Stationery':('4','2694.00')}
for row in report.collect():
    assert (str(row['orders']),str(row['revenue']))==expected[row['category']]

In [ ]:
destination = ROOT / 'output' / 'project_category_revenue_parquet'
report.write.mode('overwrite').parquet(str(destination))
loaded = spark.read.parquet(str(destination))
assert loaded.count() == 4
loaded.orderBy('category').show(truncate=False)
print('SUCCESS: project report saved at',destination)

### Reflection

How many raw rows were duplicates? Why do the two orphan transaction IDs pass the initial data-quality filter but fail the reference join? Why is Parquet suitable for an analytical summary?